In [0]:
print("Apache Spark Version:", spark.version)

Apache Spark Version: 4.2.0


In [0]:
# Vérifier la présence des 15 fichiers dans le Volume Unity Catalog
bronze_path = "/Volumes/workspace/default/crypto_bronze/Crypto_Bronze_Data"
files = dbutils.fs.ls(bronze_path)
print(f"✅ {len(files)} fichiers détectés dans la couche Bronze :")
for f in files[:5]:
    print(" -", f.name)

✅ 15 fichiers détectés dans la couche Bronze :
 - bitcoin-cash.json
 - bitcoin.json
 - cardano.json
 - chainlink.json
 - cosmos.json


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType, ArrayType
from datetime import datetime

print("🚀 Démarrage du traitement distribué Silver avec PySpark...")

# 1. Définition du Schéma Strict (Bonne pratique Big Data)
# Chaque point de série CoinGecko est un tableau de 2 éléments : [timestamp_ms, valeur]
coingecko_schema = StructType([
    StructField("prices", ArrayType(ArrayType(DoubleType())), True),
    StructField("market_caps", ArrayType(ArrayType(DoubleType())), True),
    StructField("total_volumes", ArrayType(ArrayType(DoubleType())), True)
])

# 2. Liste des fichiers JSON Bronze
bronze_dir = "/Volumes/workspace/default/crypto_bronze/Crypto_Bronze_Data"
json_files = [f.name for f in dbutils.fs.ls(bronze_dir) if f.name.endswith(".json")]

combined_df = None

for file_name in json_files:
    coin_name = file_name.replace(".json", "")
    file_path = f"{bronze_dir}/{file_name}"
    
    # Lecture avec schéma strict
    raw_df = spark.read.schema(coingecko_schema).json(file_path)
    
    # Aplatissement synchronisé avec posexplode (conserve la correspondance temporelle exacte)
    exploded_df = raw_df.select(
        F.posexplode("prices").alias("pos_p", "price_point"),
        F.col("market_caps"),
        F.col("total_volumes")
    ).select(
        F.lit(coin_name).alias("coin_name"),
        # Extraction du timestamp et conversion en Date ISO
        F.to_date(F.from_unixtime(F.col("price_point")[0] / 1000)).alias("Date"),
        F.round(F.col("price_point")[1], 6).alias("price_usd"),
        F.round(F.col("market_caps")[F.col("pos_p")][1], 2).alias("market_cap"),
        F.round(F.col("total_volumes")[F.col("pos_p")][1], 2).alias("volume")
    )
    
    if combined_df is None:
        combined_df = exploded_df
    else:
        combined_df = combined_df.unionByName(exploded_df)

# 3. Dédoublonnage et Horodatage d'Audit Technique
silver_clean_df = combined_df.dropDuplicates(["coin_name", "Date"]) \
    .filter(F.col("Date").isNotNull() & F.col("price_usd").isNotNull()) \
    .withColumn("ingestion_timestamp", F.current_timestamp())

print("📊 Aperçu des données Silver nettoyées :")
display(silver_clean_df.limit(10))

# 4. Sauvegarde au format Delta Lake dans Unity Catalog
silver_target_path = "/Volumes/workspace/default/crypto_bronze/crypto_silver_clean"
silver_clean_df.write.format("delta").mode("overwrite").save(silver_target_path)

print(f"🎉 Couche Silver écrite avec succès au format Delta Lake dans : {silver_target_path}")

🚀 Démarrage du traitement distribué Silver avec PySpark...
📊 Aperçu des données Silver nettoyées :


coin_name,Date,price_usd,market_cap,volume,ingestion_timestamp
bitcoin-cash,2025-11-03,536.969344,1.06944211347E10,2.9410678453E8,2026-10-29T21:02:39.735Z
bitcoin-cash,2025-11-27,543.808442,1.085584673962E10,4.9421004229E8,2026-10-29T21:02:39.735Z
bitcoin-cash,2025-12-27,593.590999,1.185440838738E10,3.627576111E8,2026-10-29T21:02:39.735Z
bitcoin-cash,2026-01-12,647.556624,1.293812260537E10,5.457681452E8,2026-10-29T21:02:39.735Z
bitcoin-cash,2026-01-20,583.280111,1.165589418197E10,2.7713385393E8,2026-10-29T21:02:39.735Z
bitcoin-cash,2026-02-11,524.21507,1.048192108842E10,2.4414382205E8,2026-10-29T21:02:39.735Z
bitcoin-cash,2026-02-13,504.669398,1.009047660188E10,2.0622102842E8,2026-10-29T21:02:39.735Z
bitcoin-cash,2026-03-03,448.223521,8.96549008753E9,2.3141949543E8,2026-10-29T21:02:39.735Z
bitcoin-cash,2026-03-04,448.575282,8.97258357429E9,1.6865683845E8,2026-10-29T21:02:39.735Z
bitcoin-cash,2026-03-15,469.402072,9.39153256383E9,1.2008749968E8,2026-10-29T21:02:39.735Z


🎉 Couche Silver écrite avec succès au format Delta Lake dans : /Volumes/workspace/default/crypto_bronze/crypto_silver_clean
